# Parallel Computation

## Inspecting Available Devices

In [1]:
import os

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=4"

import jax

devices = jax.devices()
n_devices = len(devices)
for i, d in enumerate(devices):
    print(f"   [{i}] {d}")

print(f"Default backend: {jax.default_backend()}")

   [0] cpu:0
   [1] cpu:1
   [2] cpu:2
   [3] cpu:3
Default backend: cpu


## `pmap`: Mapping Across Devices

In [2]:
import jax.numpy as jnp

In [3]:
def f(x):
    return x**2 + 1.0

In [4]:
pmap_f = jax.pmap(f)

x = jnp.ones((n_devices, 4))
y = pmap_f(x)

In [ ]:
print(f"Output: {y}")
print(f"Output shape: {y.shape}")
print(f"Output type: {type(y)}")

Output: [[2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]]
Output shape: (4, 4)
Output type: <class 'jaxlib._jax.ArrayImpl'>


## `axis_name` and Collective Operations

In [6]:
import functools


@functools.partial(jax.pmap, axis_name="devices")
def mean_across_devices(x):
    local_mean = jnp.mean(x)
    # jax.debug.print("local_mean {}", local_mean)
    global_mean = jax.lax.pmean(local_mean, axis_name="devices")
    return global_mean

In [7]:
x = jnp.arange(n_devices * 4, dtype=jnp.float32).reshape(n_devices, 4)

In [8]:
result = mean_across_devices(x)
print(f"Global mean (same on all devices): {result}")
print(f"Expected: {jnp.mean(x)}")

Global mean (same on all devices): [7.5 7.5 7.5 7.5]
Expected: 7.5


## A Data-Parallel Training Step

In [9]:
import jax.random as jrandom


def loss_fn(params, x, y):
    pred = params["w"] @ x + params["b"]
    return jnp.mean((pred - y) ** 2)


@functools.partial(jax.pmap, axis_name="devices")
def parallel_train_step(params, x_batch, y_batch):

    batched_loss = lambda p: jnp.mean(
        jax.vmap(loss_fn, in_axes=(None, 0, 0))(p, x_batch, y_batch)
    )
    loss, grads = jax.value_and_grad(batched_loss)(params)

    grads = jax.lax.pmean(grads, axis_name="devices")
    loss = jax.lax.pmean(loss, axis_name="devices")

    new_params = jax.tree_util.tree_map(lambda p, g: p - 0.01 * g, params, grads)
    # jax.debug.print("new_params {}", new_params)

    return new_params, loss


key = jrandom.PRNGKey(0)
k_w, k_x, k_y = jrandom.split(key, 3)
params = {"w": jrandom.normal(k_w, (2, 4)), "b": jnp.zeros(2)}

replicated_params = jax.tree_util.tree_map(
    lambda p: jnp.broadcast_to(p[None], (n_devices,) + p.shape), params
)

X = jrandom.normal(k_x, (n_devices, 16, 4))
y = jrandom.normal(k_y, (n_devices, 16, 2))

replicated_params, losses = parallel_train_step(replicated_params, X, y)
print("Loss (same on all devices):", losses)

Loss (same on all devices): [4.295835 4.295835 4.295835 4.295835]


## The Modern Sharding API

In [15]:
from jax.sharding import Mesh, PartitionSpec, NamedSharding
import numpy as np

device_array = np.array(devices)
mesh = Mesh(device_array, ("devices",))

print(f"Mesh: {mesh}")
print(f"Mesh shape: {mesh.shape}")

Mesh: Mesh('devices': 4, axis_types=(Auto,))
Mesh shape: OrderedDict({'devices': 4})


In [18]:
sharding = NamedSharding(mesh, PartitionSpec("devices", None))

x = jnp.ones((n_devices * 4, 8))
x_sharded =jax.device_put(x, sharding)

In [20]:
x_sharded.sharding

NamedSharding(mesh=Mesh('devices': 4, axis_types=(Auto,)), spec=P('devices', None), memory_kind=device)

In [21]:
x_sharded.shape

(16, 8)

## `jit` with Sharded Inputs

In [22]:
@jax.jit
def f(x, w):
    return x @ w


w = jnp.ones((8, 4))

result = f(x_sharded, w)
print(f"Result shape: {result.shape}")
print(f"Result sharding: {result.sharding}")

Result shape: (16, 4)
Result sharding: NamedSharding(mesh=Mesh('devices': 4, axis_types=(Auto,)), spec=P('devices', None), memory_kind=device)
